Sources:
(https://www.rdkit.org/docs/source/rdkit.Chem.rdMolDescriptors.html)
https://www.rdkit.org/docs/GettingStartedInPython.html#substructure-searches

In [35]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors, PandasTools
from sklearn.preprocessing import OneHotEncoder

In [39]:
#import linker data and resolve UnicodeDecodeError
linker = pd.read_csv('data/linkers.csv', encoding = 'latin1')
#convert SMILES to RDKit, remove invalid entries
linker = linker.assign(molecule = linker['smiles'].apply(Chem.MolFromSmiles)).query('molecule.notnull()')
#create canonical smiles so that all versions of same molecule maps to the same SMILES string
linker['smiles'] = linker['molecule'].apply(Chem.MolToSmiles)
#remove duplicate values

#use one hot encoding -- needed for later
linker_encoder = OneHotEncoder(sparse_output = False)
encoded_linker = linker_encoder.fit_transform(linker[['source']])
encoded = pd.DataFrame(encoded_linker, columns = linker_encoder.get_feature_names_out(['source']))
#join both the linker and OHE data
linker = pd.concat([linker.reset_index(drop = True), encoded.reset_index(drop = True)], axis = 1)
linker.head()

,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,Product name,smiles,calc_SA_score,TPSA,QED,LogP,Molecular Weight,K-means,scaffold_smiles,source,molecule,source_train,source_val
0,0,0,0,(Ac)Phe-Lys(Alloc)-PABC-PNP,C=CCOC(=O)NCCCC[C@H](NC(=O)[C@H](Cc1ccccc1)NC(...,3.508295,204.30,0.036588,4.5636,689.72,12,O=C(CCc1ccccc1)NCC(=O)Nc1ccc(COC(=O)Oc2ccccc2)cc1,train,<rdkit.Chem.rdchem.Mol object at 0x704406085000>,1.0,0.0
1,1,1,1,6-Maleimidohexanoic acid N-hydroxysuccinimide ...,O=C(CCCCCN1C(=O)C=CC1=O)ON1C(=O)CCC1=O,2.676214,101.06,0.487410,0.0790,308.29,4,O=C(CCCCCN1C(=O)C=CC1=O)ON1C(=O)CCC1=O,train,<rdkit.Chem.rdchem.Mol object at 0x704406085070>,1.0,0.0
2,2,2,2,Fmoc-Val-Cit-PAB,CC(C)[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)N...,3.338596,171.88,0.163403,3.6140,601.70,12,O=C(CNC(=O)OCC1c2ccccc2-c2ccccc21)NCC(=O)Nc1cc...,train,<rdkit.Chem.rdchem.Mol object at 0x704406000c10>,1.0,0.0
3,3,3,3,Fmoc-Val-Cit-PAB-PNP,CC(C)[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)N...,3.753739,230.32,0.030495,5.7455,766.81,12,O=C(CNC(=O)OCC1c2ccccc2-c2ccccc21)NCC(=O)Nc1cc...,train,<rdkit.Chem.rdchem.Mol object at 0x704406000ba0>,1.0,0.0
4,4,4,4,Mc-Val-Cit-PABC-PNP,CC(C)[C@H](NC(=O)CCCCCN1C(=O)C=CC1=O)C(=O)N[C@...,3.796268,258.47,0.032958,2.8084,737.77,12,O=C(CCCCCN1C(=O)C=CC1=O)NCC(=O)NCC(=O)Nc1ccc(C...,train,<rdkit.Chem.rdchem.Mol object at 0x704406000b30>,1.0,0.0


In [40]:
#get the SMARTS syntax for hydrophilic functional groups -- going to try to find more that are important
hg = {'hydroxyl': Chem.MolFromSmarts('[OX2;H]'), 'carboxyl' : Chem.MolFromSmarts('C(=O)[OX2H]'), 'amino': Chem.MolFromSmarts('[NX3;H2;!$(NC=O)]'), 'phosphate': Chem.MolFromSmarts('[PX4;!$(P(=O)(O)(O))]')}

def smarts(func_group):
    
    valid_smarts = {}
    for name, molecule in func_group.items():
        if molecule:
            valid_smarts[name] = molecule
        else:
            print(f'Invalid')
        
    return valid_smarts

#use SMARTS on a molecule
def run_smarts (molecule, pattern):

    return {name: int(molecule.HasSubstructMatch(p)) for name, p in pattern.items()}

#apply to all
hydrophilic_groups = linker['molecule'].apply(lambda mol: run_smarts(mol, hg))

#create DataFrame
hydrophilic = pd.DataFrame(list(hydrophilic_groups), index = linker.index)
#when this re-runs, duplicate columns are made.. this will prevent those from popping up
linker = linker.drop(columns = ['hydroxyl', 'carboxyl', 'amino', 'phosphate'], errors = 'ignore')
#join both dataframes
linker = linker.join(hydrophilic)

linker.head()


,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,Product name,smiles,calc_SA_score,TPSA,QED,LogP,Molecular Weight,K-means,scaffold_smiles,source,molecule,source_train,source_val,hydroxyl,carboxyl,amino,phosphate
0,0,0,0,(Ac)Phe-Lys(Alloc)-PABC-PNP,C=CCOC(=O)NCCCC[C@H](NC(=O)[C@H](Cc1ccccc1)NC(...,3.508295,204.30,0.036588,4.5636,689.72,12,O=C(CCc1ccccc1)NCC(=O)Nc1ccc(COC(=O)Oc2ccccc2)cc1,train,<rdkit.Chem.rdchem.Mol object at 0x704406085000>,1.0,0.0,0,0,0,0
1,1,1,1,6-Maleimidohexanoic acid N-hydroxysuccinimide ...,O=C(CCCCCN1C(=O)C=CC1=O)ON1C(=O)CCC1=O,2.676214,101.06,0.487410,0.0790,308.29,4,O=C(CCCCCN1C(=O)C=CC1=O)ON1C(=O)CCC1=O,train,<rdkit.Chem.rdchem.Mol object at 0x704406085070>,1.0,0.0,0,0,0,0
2,2,2,2,Fmoc-Val-Cit-PAB,CC(C)[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)N...,3.338596,171.88,0.163403,3.6140,601.70,12,O=C(CNC(=O)OCC1c2ccccc2-c2ccccc21)NCC(=O)Nc1cc...,train,<rdkit.Chem.rdchem.Mol object at 0x704406000c10>,1.0,0.0,1,0,0,0
3,3,3,3,Fmoc-Val-Cit-PAB-PNP,CC(C)[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)N...,3.753739,230.32,0.030495,5.7455,766.81,12,O=C(CNC(=O)OCC1c2ccccc2-c2ccccc21)NCC(=O)Nc1cc...,train,<rdkit.Chem.rdchem.Mol object at 0x704406000ba0>,1.0,0.0,0,0,0,0
4,4,4,4,Mc-Val-Cit-PABC-PNP,CC(C)[C@H](NC(=O)CCCCCN1C(=O)C=CC1=O)C(=O)N[C@...,3.796268,258.47,0.032958,2.8084,737.77,12,O=C(CCCCCN1C(=O)C=CC1=O)NCC(=O)NCC(=O)Nc1ccc(C...,train,<rdkit.Chem.rdchem.Mol object at 0x704406000b30>,1.0,0.0,0,0,0,0


In [41]:
linker['total_rings'] = linker['molecule'].apply(lambda mol: mol.GetRingInfo().NumRings())
linker['aromatic_rings'] = linker['molecule'].apply(lambda mol: Descriptors.NumAromaticRings(mol))
linker['aliphatic_rings'] = linker['molecule'].apply(lambda mol: Descriptors.NumAliphaticRings(mol))

In [42]:
linker.head()

,Unnamed: 0.2,Unnamed: 0,Unnamed: 0.1,Product name,smiles,calc_SA_score,TPSA,QED,LogP,Molecular Weight,...,molecule,source_train,source_val,hydroxyl,carboxyl,amino,phosphate,total_rings,aromatic_rings,aliphatic_rings
0,0,0,0,(Ac)Phe-Lys(Alloc)-PABC-PNP,C=CCOC(=O)NCCCC[C@H](NC(=O)[C@H](Cc1ccccc1)NC(...,3.508295,204.30,0.036588,4.5636,689.72,...,<rdkit.Chem.rdchem.Mol object at 0x704406085000>,1.0,0.0,0,0,0,0,3,3,0
1,1,1,1,6-Maleimidohexanoic acid N-hydroxysuccinimide ...,O=C(CCCCCN1C(=O)C=CC1=O)ON1C(=O)CCC1=O,2.676214,101.06,0.487410,0.0790,308.29,...,<rdkit.Chem.rdchem.Mol object at 0x704406085070>,1.0,0.0,0,0,0,0,2,0,2
2,2,2,2,Fmoc-Val-Cit-PAB,CC(C)[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)N...,3.338596,171.88,0.163403,3.6140,601.70,...,<rdkit.Chem.rdchem.Mol object at 0x704406000c10>,1.0,0.0,1,0,0,0,4,3,1
3,3,3,3,Fmoc-Val-Cit-PAB-PNP,CC(C)[C@H](NC(=O)OCC1c2ccccc2-c2ccccc21)C(=O)N...,3.753739,230.32,0.030495,5.7455,766.81,...,<rdkit.Chem.rdchem.Mol object at 0x704406000ba0>,1.0,0.0,0,0,0,0,5,4,1
4,4,4,4,Mc-Val-Cit-PABC-PNP,CC(C)[C@H](NC(=O)CCCCCN1C(=O)C=CC1=O)C(=O)N[C@...,3.796268,258.47,0.032958,2.8084,737.77,...,<rdkit.Chem.rdchem.Mol object at 0x704406000b30>,1.0,0.0,0,0,0,0,3,2,1


In [10]:
linker.columns

Index(['Unnamed: 0.2', 'Unnamed: 0', 'Unnamed: 0.1', 'Product name', 'smiles',
       'calc_SA_score', 'TPSA', 'QED', 'LogP', 'Molecular Weight', 'K-means',
       'scaffold_smiles', 'source', 'molecule', 'hydroxyl', 'carboxyl',
       'amino', 'phosphate'],
      dtype='object')

__calc_SA_score:__ SAS measures difficulty of synthesizing a chemical molecule, SA rates the ease of synthesis of molecule on scale of 1-10 from easiest to hardest. Good benchmark for generated structures.<br>
__TPSA:__ the sum of surface area over all polar atoms, measures the drugs ability to permeate membranes. Greater than 140 means poor at permeating cell membranes.<br>
__QED:__ Quantitative estimate of drug-likeness. Incorporated as an optimization metric, trying to maximize drug likeness of linkers.<br>
__LogP:__ logarithm of hte partition coefficient, compares solubility of the solute in two immiscible solvents. If one is water and the other non-polar then this is a measure of hydrophobicity.<br>
__Ring Number:__ the number of rings in a compound.<br>
__K-means:__ 